<!--nav--> [🗺 Learning path](README.md) · **21/30** · ◀ [Simple MultiGPU Audio](./Simple_MultiGPU_Audio.ipynb) · [vLLM High-Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) ▶

# LLM Serving Fundamentals: KV Cache, Prefill/Decode & Continuous Batching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Serving_Fundamentals_KV_Cache_Batching.ipynb)

You've trained a model (notebooks 1–20). Now someone wants to *use* it — and suddenly you care about
completely different things: **latency**, **throughput**, and **how many users fit on one GPU**.

This notebook builds the mental model that every modern serving engine (vLLM, SGLang, TensorRT-LLM)
is designed around. Everything is measured live — no numbers on faith.

| Part | What you'll learn | What you'll measure |
|---|---|---|
| **0** | Why inference has *two phases* (prefill / decode) with opposite bottlenecks | — |
| **1** | The KV cache: what it stores, the memory formula, GQA vs MHA vs MLA | KV bytes/token for real models; with vs without cache |
| **2** | The three numbers that define serving: TTFT, TPOT, throughput | TTFT vs prompt length; per-token decode time |
| **3** | Batching: why decode batching is almost free, and why *static* batching wastes GPUs | tokens/s at batch 1→16; a continuous-batching simulator |

**Runs on:** free Colab **T4** (recommended) · Parts 1a and 3b are pure Python and run anywhere, even CPU.

**Model:** `Qwen/Qwen2.5-0.5B-Instruct` — small enough to download in seconds, real enough that every
effect we measure also holds for 70B models (just with bigger constants).

## Part 0 · One request, two phases

A single chat completion is really **two different workloads** glued together:

```
 PREFILL (a.k.a. "prompt processing")          DECODE (a.k.a. "generation")
 ──────────────────────────────────     ────────────────────────────────
 "What is the capital of France?"              "The" → " capital" → " is" → " Paris" → ...
   all N prompt tokens processed                 ONE token per forward pass,
   in ONE big parallel forward pass              each pass re-reads ALL the weights

 ✅ Big matrix multiplies                      ❌ Matrix-VECTOR multiplies
 ✅ GPU compute units saturated                ❌ GPU waits on memory traffic
 → COMPUTE-bound                               → MEMORY-BANDWIDTH-bound
```

**Why decode is memory-bound — the 30-second argument.** To produce *one* token, the GPU must read
every weight of the model from HBM once (~1GB for our 0.5B model in fp16, ~14GB for a 7B). The
arithmetic done per weight read is tiny (one multiply-add per weight for batch size 1). A T4 can do
~65 TFLOP/s (fp16) but only move ~300 GB/s. At batch 1 you use maybe **1–2% of the compute** while
the memory bus runs flat out. The compute sits idle.

That single fact explains almost everything in modern serving:

- **Batching decode is nearly free** — 8 requests share one weight-read → ~8× throughput (Part 3).
- **The KV cache exists** so decode does O(1) work per token instead of re-running the whole prompt (Part 1).
- **Speculative decoding exists** to spend the idle compute on guessing future tokens (notebook 24).
- **Prefill and decode fight each other** when scheduled on the same GPU — which is why the frontier
  is *disaggregating* them onto different machines (notebook 24).

In [ ]:
# Setup — works on Colab T4 (recommended) or any machine with recent transformers.
# Parts 1a & 3b are pure Python; the measurement cells want a GPU.
!pip install -q -U transformers accelerate

import torch, time
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/1e9:.1f} GB · compute capability {p.major}.{p.minor}")
else:
    print("No GPU found — the calculator & simulator cells still run; measurement cells will be slow.")

## Part 1 · The KV cache — the thing you're actually serving

Attention at step *t* needs the **K**ey and **V**alue vectors of *every previous token*. Without a
cache you'd recompute them all from scratch for every new token — O(n²) work per token, O(n³) per
sequence. The KV cache stores them once, making each decode step O(n) reads and O(1) new compute.

The price is **memory**, and it's the commodity a serving engine actually manages:

```
                       (K and V)
KV bytes per token  =  2 × n_layers × n_kv_heads × head_dim × bytes_per_element
```

Three architecture tricks exist purely to shrink this number:

| Scheme | Idea | Who uses it |
|---|---|---|
| **MHA** (multi-head) | every query head has its own K/V head | GPT-3 era |
| **GQA** (grouped-query) | groups of query heads *share* one K/V head (e.g. 14 Q heads → 2 KV heads) | Llama 3, Qwen 2.5, Mistral |
| **MQA** (multi-query) | ALL query heads share 1 KV head | Falcon, PaLM |
| **MLA** (multi-head latent) | store a low-rank *compressed* latent instead of K/V; decompress on the fly | DeepSeek V2/V3/R1 |

Let's compute real numbers — this cell is pure Python and runs anywhere:

In [ ]:
# KV-cache calculator for real, current model architectures (from their HF config.json files).
# (layers, n_kv_heads, head_dim, note) — MLA stores a compressed latent instead, handled below.
MODELS = {
    "Qwen2.5-0.5B  (GQA)":  dict(layers=24,  kv_heads=2,  head_dim=64,  weights_gb=1.0),
    "Llama-3.1-8B  (GQA)":  dict(layers=32,  kv_heads=8,  head_dim=128, weights_gb=16.1),
    "Qwen2.5-72B   (GQA)":  dict(layers=80,  kv_heads=8,  head_dim=128, weights_gb=145.0),
    "GPT-3-175B    (MHA)":  dict(layers=96,  kv_heads=96, head_dim=128, weights_gb=350.0),
    "DeepSeek-V3   (MLA)":  dict(layers=61,  kv_heads=None, head_dim=None, weights_gb=1342.0,
                                 mla_latent=512 + 64),  # compressed KV latent + decoupled RoPE dims
}
BYTES = 2  # fp16/bf16 cache

print(f"{'model':<22}{'KV KB/token':>12}{'KV for 8k ctx':>15}{'8k seqs in 16GB*':>18}")
print("-" * 67)
for name, m in MODELS.items():
    if m.get("mla_latent"):
        per_tok = m["layers"] * m["mla_latent"] * BYTES          # MLA: one latent vector per layer
    else:
        per_tok = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * BYTES
    kb = per_tok / 1024
    gb_8k = per_tok * 8192 / 1e9
    # * toy scenario: 16 GB card, pretending weights fit elsewhere — isolates the KV pressure
    fits = int(16e9 // (per_tok * 8192))
    print(f"{name:<22}{kb:>10.1f}KB{gb_8k:>13.2f}GB{fits:>18}")

print("\nGPT-3-era MHA: ~4.7 MB per TOKEN — ~39 GB of cache for ONE 8k conversation.")
print("GQA cut that ~12x; MLA cuts another ~2x on top — KV memory is a first-class design axis.")

**Read that table again — it's the whole serving problem in four rows.** A GPT-3-class MHA model
supports a *handful* of long conversations per GPU before KV memory runs out. GQA models support
dozens. This is why every 2024+ model ships with GQA or MLA: **KV bytes/token is a product decision**,
it determines your cost per user.

### The other half of the problem: *where* the cache lives

Knowing the size isn't enough. Classic HF-style serving allocates each request a **contiguous**
buffer sized for `max_len` — even if the request ends after 30 tokens:

```
Request A (asked for 4096, used 300):  [███..............................]  ← ~93% wasted
Request B (asked for 4096, used 3900): [██████████████████████████████.]
                gaps between buffers → external fragmentation → can't start new requests
```

The vLLM paper measured **60–80% of KV memory wasted** in systems like this. Their fix —
**PagedAttention**, virtual-memory-style paging for the KV cache — is the subject of the next
notebook. Here, let's first *prove the cache matters at all*:

In [ ]:
# Measure: generation WITH vs WITHOUT the KV cache (the O(n) vs O(n^2) difference, live).
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE).eval()

def chat_ids(prompt):
    # two-step on purpose: works across transformers v4 and v5
    # (v5's apply_chat_template(return_tensors=...) returns a BatchEncoding, not a tensor)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   add_generation_prompt=True, tokenize=False)
    return tok(text, return_tensors="pt").input_ids.to(DEVICE)

def timed_generate(n_new, use_cache, prompt="Explain what a hash table is, in detail."):
    ids = chat_ids(prompt)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=n_new, min_new_tokens=n_new,
                             do_sample=False, use_cache=use_cache,
                             pad_token_id=tok.eos_token_id)
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return dt, (out.shape[1] - ids.shape[1]) / dt

timed_generate(8, True)  # warmup (kernel compilation, memory pools)

print(f"{'new tokens':>10} {'with cache':>14} {'without cache':>15} {'slowdown':>9}")
for n in (32, 64, 128):
    t_c, tps_c = timed_generate(n, True)
    t_n, tps_n = timed_generate(n, False)
    print(f"{n:>10} {t_c:>9.2f}s ({tps_c:>4.0f}/s) {t_n:>9.2f}s ({tps_n:>4.0f}/s) {t_n/t_c:>8.1f}x")

print("\nNo cache => every token re-processes the ENTIRE sequence so far.")
print("The gap widens as generation gets longer — that's the O(n²) showing up.")

## Part 2 · The three numbers on every serving dashboard

| Metric | Definition | Dominated by | User feels it as |
|---|---|---|---|
| **TTFT** | Time To First Token | **prefill** (scales with prompt length) | “is it thinking?” |
| **TPOT / ITL** | Time Per Output Token (inter-token latency) | **decode** (≈ constant per token) | streaming speed |
| **Throughput** | total tokens/s across *all* users | batching efficiency | your GPU bill |

Total request latency = `TTFT + TPOT × output_tokens`.

The tension: **batching more requests raises throughput but hurts each request's TPOT slightly** —
serving is the art of trading those off. (SLO-aware schedulers call the sweet spot “goodput”:
requests/s that *meet* the latency target, not just raw tokens/s.)

Let's confirm the two claims baked into that table — TTFT grows with prompt length, TPOT doesn't:

In [ ]:
# Measure TTFT (prefill cost) and TPOT (decode cost) as the PROMPT gets longer.
filler = ("The history of computing is a story of layers of abstraction, "
          "each hiding the complexity of the one below it. ")

def ttft_tpot(prompt_tokens, n_new=64):
    text = filler * 200
    ids = tok(text, return_tensors="pt", truncation=True, max_length=prompt_tokens).input_ids.to(DEVICE)
    with torch.no_grad():
        if DEVICE == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.generate(ids, max_new_tokens=1, do_sample=False, pad_token_id=tok.eos_token_id)
        if DEVICE == "cuda": torch.cuda.synchronize()
        ttft = time.perf_counter() - t0                      # prefill + 1 token
        t0 = time.perf_counter()
        model.generate(ids, max_new_tokens=n_new, min_new_tokens=n_new,
                       do_sample=False, pad_token_id=tok.eos_token_id)
        if DEVICE == "cuda": torch.cuda.synchronize()
        total = time.perf_counter() - t0
    tpot = (total - ttft) / (n_new - 1)                      # marginal per-token decode time
    return ttft, tpot

ttft_tpot(32)  # warmup
print(f"{'prompt tokens':>13} {'TTFT':>9} {'TPOT':>9}")
for n in (32, 128, 512, 1024, 2048):
    ttft, tpot = ttft_tpot(n)
    print(f"{n:>13} {ttft*1000:>7.0f}ms {tpot*1000:>7.1f}ms")

print("\nTTFT climbs with prompt length (prefill does real work per prompt token).")
print("TPOT stays ~flat (each decode step reads the same weights regardless of history length —")
print("the KV-cache reads grow, but for short contexts the weight reads dominate).")

## Part 3 · Batching — where the 10× lives

Decode at batch 1 wastes ~98% of the GPU's compute (Part 0). The fix is obvious in hindsight:
**make one weight-read serve many requests**. Reading 1 GB of weights to produce 1 token is a bad
deal; reading 1 GB to produce 16 tokens (one for each of 16 users) is a 16× better one — and the
memory bus, our bottleneck, does *almost the same work either way*.

First, measure how far that logic carries on real hardware:

In [ ]:
# Static-batch decode throughput: same prompt replicated B times, generate 64 tokens each.
tok.padding_side = "left"                       # decoder-only models pad on the left for generation
if tok.pad_token is None: tok.pad_token = tok.eos_token

def batch_tps(bsz, n_new=64):
    prompts = ["Write a short poem about the sea."] * bsz
    msgs = [tok.apply_chat_template([{"role":"user","content":p}], add_generation_prompt=True, tokenize=False)
            for p in prompts]
    enc = tok(msgs, return_tensors="pt", padding=True).to(DEVICE)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**enc, max_new_tokens=n_new, min_new_tokens=n_new,
                       do_sample=False, pad_token_id=tok.eos_token_id)
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return bsz * n_new / dt, dt / n_new * 1000   # total tokens/s, per-step ms

batch_tps(1, 16)  # warmup
print(f"{'batch':>5} {'tokens/s':>10} {'ms/step':>9} {'scaling':>9}")
base = None
for b in (1, 2, 4, 8, 16):
    tps, ms = batch_tps(b)
    base = base or tps
    print(f"{b:>5} {tps:>10.0f} {ms:>9.1f} {tps/base:>8.1f}x")

print("\nThroughput scales almost linearly while ms/step barely moves:")
print("batched decode rides along with the weight reads we were already paying for.")

### So just batch everything? The straggler problem

**Static batching** (what we just did, and what `model.generate` does) has a fatal flaw for serving:
the batch is formed *once* and nobody leaves until **everyone** is finished.

```
STATIC BATCH (batch=4, █ = generating, · = done but stuck holding its slot)

req A: ██████····················   finished at t=6, waits for D
req B: ████████████···············   finished at t=12
req C: ███························   finished at t=3 (!)
req D: ███████████████████████████   the straggler defines everyone's latency
       ↑ meanwhile NEW requests queue outside, GPU slots sit idle-but-occupied

CONTINUOUS BATCHING (iteration-level scheduling — Orca 2022, then vLLM)

req A: ██████│E █████████████...       the moment A finishes, E takes its slot
req C: ███│F ████████████████...       scheduler re-decides EVERY iteration
```

Every modern engine (vLLM, SGLang, TensorRT-LLM, TGI) schedules **per iteration**, not per batch.
New requests join mid-flight (their prefill gets chunked in between decode steps), finished ones
leave instantly. Let's quantify what that's worth with a simulator — pure Python, runs anywhere:

In [ ]:
# Continuous vs static batching — a discrete-time simulator.
# One time-step = one decode iteration for everything currently in the batch.
import random
random.seed(0)

N_REQ, SLOTS = 200, 8
# Realistic long-tail output lengths: most replies short, some very long
lengths = [min(512, max(4, int(random.lognormvariate(3.6, 0.9)))) for _ in range(N_REQ)]

def simulate(policy):
    queue = list(range(N_REQ)); active = {}; done = {}; t = 0; busy_slot_steps = 0
    while queue or active:
        if policy == "static" and not active:            # refill only when the whole batch drained
            for _ in range(min(SLOTS, len(queue))):
                r = queue.pop(0); active[r] = lengths[r]
        elif policy == "continuous":                     # refill every iteration
            while len(active) < SLOTS and queue:
                r = queue.pop(0); active[r] = lengths[r]
        t += 1
        busy_slot_steps += len(active)
        for r in list(active):
            active[r] -= 1
            if active[r] == 0: done[r] = t; del active[r]
    total_tokens = sum(lengths)
    return dict(makespan=t,
                throughput=total_tokens / t,
                avg_completion=sum(done.values()) / N_REQ,
                slot_util=busy_slot_steps / (t * SLOTS))

print(f"{'policy':<12}{'total steps':>12}{'tokens/step':>13}{'avg finish t':>14}{'slot util':>11}")
for pol in ("static", "continuous"):
    s = simulate(pol)
    print(f"{pol:<12}{s['makespan']:>12}{s['throughput']:>13.2f}{s['avg_completion']:>14.0f}{s['slot_util']:>10.0%}")

print(f"\n(200 requests, {SLOTS} slots, lognormal output lengths 4..512 — the long tail is the point.)")
print("Static batching idles slots waiting for each batch's straggler; continuous batching back-fills")
print("them instantly. Same GPU, same requests — the scheduler alone buys the difference.")

### What the simulator just showed you

With a realistic long-tailed length distribution, continuous batching typically delivers
**~1.5–2× the throughput** and dramatically better average completion time — *purely from
scheduling*, before any kernel or memory tricks. Stack the real-world effects on top
(new requests keep arriving; static batching also pads every sequence to the batch max) and you get
the **up-to-23× throughput** gains the vLLM/Orca line of work reported over naive serving.

## Recap — the serving mental model

1. **Decode is memory-bandwidth-bound** → batch it; the weight-read is shared. (measured: ~linear scaling to 16)
2. **The KV cache turns O(n²) into O(n)** → but its *memory* becomes the scarce resource you schedule. (measured: 2–4×+ and growing)
3. **TTFT is prefill, TPOT is decode** → they scale differently and want different optimizations. (measured)
4. **Iteration-level (continuous) batching** → no straggler tax, slots never idle. (simulated: ~2×)

Every technique in the next three notebooks attacks one of these four:

| Bottleneck | Attack | Notebook |
|---|---|---|
| KV memory waste | **PagedAttention**, prefix caching | [vLLM High Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) |
| Weight-read bandwidth | **Quantization** (AWQ/GPTQ/FP8) | [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) |
| Idle compute during decode | **Speculative decoding** | [Speculative Decoding & the Serving Frontier](./Speculative_Decoding_Advanced_Serving.ipynb) |

### Further reading
- [Efficient Memory Management for LLM Serving with PagedAttention](https://arxiv.org/abs/2309.06180) (the vLLM paper)
- [Orca: A Distributed Serving System for Transformer-Based Generative Models](https://www.usenix.org/conference/osdi22/presentation/yu) (continuous batching, OSDI '22)
- [GQA: Training Generalized Multi-Query Transformer Models](https://arxiv.org/abs/2305.13245)
- [DeepSeek-V2](https://arxiv.org/abs/2405.04434) (introduces MLA, §2.1 is the readable part)
- Kipply's [Transformer Inference Arithmetic](https://kipp.ly/transformer-inference-arithmetic/) — the classic back-of-envelope guide

▶ **Next:** [vLLM High Throughput Serving](./vLLM_High_Throughput_Serving.ipynb) — run the engine that turned this notebook into a product.